# Phase 19 — Unsupervised Target-Prior Correction (BBSE)

Goal: improve UNSW-NB15 → CICIDS2017 cross-dataset performance without using CICIDS labels during adaptation. This phase freezes a deterministic target adaptation/evaluation split, fits the existing CORAL + balanced Random Forest benchmark, estimates target class prior from unlabeled CICIDS predictions using Black Box Shift Estimation (BBSE), and corrects posterior probabilities. CICIDS labels are used only for final evaluation/post-hoc diagnostics.

In [ ]:
import gc, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

RANDOM_STATE=42; CHUNK=100_000; TARGET_POOL_N=600_000; TARGET_ADAPT_N=300_000; TARGET_EVAL_N=300_000
ROOT=Path.cwd(); OUT=ROOT/"outputs"/"phase19_bbse"; TABLES=OUT/"tables"; PLOTS=OUT/"plots"; TABLES.mkdir(parents=True,exist_ok=True); PLOTS.mkdir(parents=True,exist_ok=True)
FEATURES=["flow_duration_sec","fwd_packets","bwd_packets","fwd_bytes","bwd_bytes","total_packets","total_bytes","packet_rate","fwd_packet_rate","bwd_packet_rate","avg_packet_size"]

def norm(x): return str(x).replace("\ufeff","").replace("\xa0"," ").strip().lower()
def header(path):
    h=pd.read_csv(path,nrows=0,low_memory=True); h.columns=[str(c).replace("\ufeff","").replace("\xa0"," ").strip() for c in h.columns]; return h.columns

search_roots=[]
for p in [ROOT,ROOT.parent,ROOT.parent.parent,ROOT.parent.parent.parent]:
    if p.exists() and p not in search_roots: search_roots.append(p)
all_csv=[]; seen=set()
for base in search_roots:
    try:
        for p in base.rglob("*.csv"):
            rp=p.resolve()
            if rp not in seen: seen.add(rp); all_csv.append(p)
    except (PermissionError,OSError): pass

print("Search roots:"); [print("  ",p) for p in search_roots]; print("CSV files discovered:",len(all_csv))

cic_required={"label","flow duration","total fwd packets","total backward packets","total length of fwd packets","total length of bwd packets","flow packets/s","fwd packets/s","bwd packets/s","average packet size"}
cicids_files=[]
for p in sorted(all_csv,key=lambda x:str(x).lower()):
    try:
        if cic_required.issubset({norm(c) for c in header(p)}): cicids_files.append(p)
    except Exception: pass

unsw_required={"label","dur","spkts","dpkts","sbytes","dbytes"}; unsw_csv=[]
for p in sorted(all_csv,key=lambda x:str(x).lower()):
    try:
        if unsw_required.issubset({norm(c) for c in header(p)}): unsw_csv.append(p)
    except Exception: pass

if not cicids_files: raise FileNotFoundError("Could not automatically locate CICIDS2017 CSV files. Run from the project folder or set CICIDS_FILES manually.")
if not unsw_csv: raise FileNotFoundError("Could not automatically locate UNSW-NB15 CSV files. Run from the project folder or set UNSW train/test paths manually.")

def score_unsw(p,kind):
    n=norm(p.name).replace(" ","").replace("_","").replace("-",""); return (10 if (kind=="train" and "training" in n) or (kind=="test" and "testing" in n) else 0)+(5 if (kind=="train" and "train" in n) or (kind=="test" and "test" in n) else 0)
UNSW_TRAIN_PATH=max(unsw_csv,key=lambda p:(score_unsw(p,"train"),-len(str(p))))
UNSW_TEST_PATH=max(unsw_csv,key=lambda p:(score_unsw(p,"test"),-len(str(p))))
if UNSW_TRAIN_PATH.resolve()==UNSW_TEST_PATH.resolve():
    ranked=sorted(unsw_csv,key=lambda p:score_unsw(p,"train"),reverse=True); UNSW_TRAIN_PATH=ranked[0]
    ranked_test=sorted(unsw_csv,key=lambda p:score_unsw(p,"test"),reverse=True); UNSW_TEST_PATH=next((p for p in ranked_test if p.resolve()!=UNSW_TRAIN_PATH.resolve()),None)
    if UNSW_TEST_PATH is None: raise FileNotFoundError("Could not identify separate UNSW training/testing files.")

print("\n========== DATASETS FOUND ==========")
print("CICIDS files:"); [print("  ",p) for p in cicids_files]
print("\nUNSW training:",UNSW_TRAIN_PATH); print("UNSW testing :",UNSW_TEST_PATH)


### Path-discovery fix
This version searches the notebook/project directory and nearby parent directories for the actual CICIDS2017 and UNSW-NB15 CSV files using their column schemas, instead of assuming `data/raw/CICIDS2017`.


In [ ]:
# Locate UNSW official train/test files
cands=[]
for p in sorted(UNSW_ROOT.rglob("*.csv")):
    try:
        h=pd.read_csv(p,nrows=0)
        if label_col(h.columns):
            n=p.name.lower().replace(" ",""); score=(5 if "training" in n else 0)+(5 if "testing" in n else 0)+(2 if "train" in n else 0)+(2 if "test" in n else 0); cands.append((score,p))
    except: pass
cands.sort(key=lambda z:(-z[0],str(z[1])))
def pick(kind):
    keys=["training-set","trainingset","training","train"] if kind=="train" else ["testing-set","testingset","testing","test"]
    for k in keys:
        for _,p in cands:
            if k in p.name.lower().replace(" ",""): return p
UNSW_TRAIN_PATH=pick("train"); UNSW_TEST_PATH=pick("test")
if UNSW_TRAIN_PATH is None or UNSW_TEST_PATH is None: raise FileNotFoundError("Could not identify UNSW training/testing CSVs. Set UNSW_TRAIN_PATH and UNSW_TEST_PATH manually.")
print("UNSW train:",UNSW_TRAIN_PATH); print("UNSW test:",UNSW_TEST_PATH)

In [ ]:
def acol(cols,name):
    r=norm(name); return next((c for c in cols if norm(c)==r),None)
def div(a,b):
    a=pd.to_numeric(a,errors="coerce"); b=pd.to_numeric(b,errors="coerce").replace(0,np.nan); return a/b
def unsw_common(df):
    m={norm(c):c for c in df.columns}; req=["dur","spkts","dpkts","sbytes","dbytes"]; miss=[x for x in req if x not in m]
    if miss: raise KeyError(f"UNSW common columns missing: {miss}")
    x=pd.DataFrame(index=df.index); x["flow_duration_sec"]=pd.to_numeric(df[m["dur"]],errors="coerce"); x["fwd_packets"]=pd.to_numeric(df[m["spkts"]],errors="coerce"); x["bwd_packets"]=pd.to_numeric(df[m["dpkts"]],errors="coerce"); x["fwd_bytes"]=pd.to_numeric(df[m["sbytes"]],errors="coerce"); x["bwd_bytes"]=pd.to_numeric(df[m["dbytes"]],errors="coerce"); x["total_packets"]=x.fwd_packets+x.bwd_packets; x["total_bytes"]=x.fwd_bytes+x.bwd_bytes; x["packet_rate"]=div(x.total_packets,x.flow_duration_sec); x["fwd_packet_rate"]=div(x.fwd_packets,x.flow_duration_sec); x["bwd_packet_rate"]=div(x.bwd_packets,x.flow_duration_sec); x["avg_packet_size"]=div(x.total_bytes,x.total_packets); return x[FEATURES]
def cic_common(df):
    m={norm(c):c for c in df.columns}; req=["flow duration","total fwd packets","total backward packets","total length of fwd packets","total length of bwd packets","flow packets/s","fwd packets/s","bwd packets/s","average packet size","label"]; miss=[x for x in req if x not in m]
    if miss: raise KeyError(f"CICIDS semantic columns missing: {miss}")
    def n(s): return pd.to_numeric(df[m[s]],errors="coerce")
    x=pd.DataFrame(index=df.index); x["flow_duration_sec"]=n("flow duration")/1e6; x["fwd_packets"]=n("total fwd packets"); x["bwd_packets"]=n("total backward packets"); x["fwd_bytes"]=n("total length of fwd packets"); x["bwd_bytes"]=n("total length of bwd packets"); x["total_packets"]=x.fwd_packets+x.bwd_packets; x["total_bytes"]=x.fwd_bytes+x.bwd_bytes; x["packet_rate"]=n("flow packets/s"); x["fwd_packet_rate"]=n("fwd packets/s"); x["bwd_packet_rate"]=n("bwd packets/s"); x["avg_packet_size"]=n("average packet size"); x=x.replace([np.inf,-np.inf],np.nan);
    for c in FEATURES: x.loc[x[c]<0,c]=np.nan
    return x[FEATURES]
tr=pd.read_csv(UNSW_TRAIN_PATH,low_memory=False); X_unsw=unsw_common(tr); y_unsw=pd.to_numeric(tr[acol(tr.columns,"label")],errors="coerce").fillna(0).astype("int8").to_numpy(); del tr; gc.collect(); print("UNSW",X_unsw.shape,np.bincount(y_unsw,minlength=2))

In [ ]:
# Memory-safe CICIDS pool. Each file is sampled proportionally.
CIC_USE=["Flow Duration","Total Fwd Packets","Total Backward Packets","Total Length of Fwd Packets","Total Length of Bwd Packets","Flow Packets/s","Fwd Packets/s","Bwd Packets/s","Average Packet Size","Label"]
counts={}
for p in cicids_files:
    h=pd.read_csv(p,nrows=0); lc=label_col(h.columns); n=0
    for ch in pd.read_csv(p,usecols=[lc],chunksize=CHUNK,low_memory=True): n+=len(ch)
    counts[p]=n; print("counted",p.name,n)
total=sum(counts.values()); target=min(TARGET_POOL_N,total); quotas={}; rem=target
for i,p in enumerate(cicids_files):
    q=rem if i==len(cicids_files)-1 else min(int(round(target*counts[p]/total)),rem); quotas[p]=q; rem-=q
parts=[]
for p in cicids_files:
    h=pd.read_csv(p,nrows=0); m={norm(c):c for c in h.columns}; use=[m[norm(c)] for c in CIC_USE]; chunks=[]
    for ch in pd.read_csv(p,usecols=use,chunksize=CHUNK,low_memory=True):
        mm={norm(c):c for c in ch.columns}; y=ch[mm["label"]].astype(str).str.strip().str.upper().ne("BENIGN").astype("int8").to_numpy(); x=cic_common(ch); x["label_binary"]=y; chunks.append(x)
    df=pd.concat(chunks,ignore_index=True); del chunks; q=min(quotas[p],len(df));
    if len(df)>q: df=df.sample(q,random_state=RANDOM_STATE).reset_index(drop=True)
    parts.append(df); print("sampled",p.name,len(df)); del df; gc.collect()
cicids_pool=pd.concat(parts,ignore_index=True); del parts; gc.collect(); print("pool",cicids_pool.shape,cicids_pool.label_binary.value_counts().to_dict())

In [ ]:
# Freeze deterministic target adaptation/evaluation partitions
idx=np.arange(len(cicids_pool)); adapt_idx,eval_idx=train_test_split(idx,test_size=TARGET_EVAL_N,random_state=RANDOM_STATE,stratify=cicids_pool.label_binary)
if len(adapt_idx)>TARGET_ADAPT_N: adapt_idx,_=train_test_split(adapt_idx,test_size=len(adapt_idx)-TARGET_ADAPT_N,random_state=RANDOM_STATE,stratify=cicids_pool.iloc[adapt_idx].label_binary)
X_cic=cicids_pool[FEATURES].copy(); y_cic=cicids_pool.label_binary.to_numpy("int8"); X_adapt=X_cic.iloc[adapt_idx].reset_index(drop=True); X_eval=X_cic.iloc[eval_idx].reset_index(drop=True); y_adapt_hidden=y_cic[adapt_idx]; y_eval=y_cic[eval_idx]
fit_idx,val_idx=train_test_split(np.arange(len(X_unsw)),test_size=.20,random_state=RANDOM_STATE,stratify=y_unsw); X_fit=X_unsw.iloc[fit_idx].reset_index(drop=True); X_val=X_unsw.iloc[val_idx].reset_index(drop=True); y_fit=y_unsw[fit_idx]; y_val=y_unsw[val_idx]
imp=SimpleImputer(strategy="median"); X_fit_i=imp.fit_transform(X_fit).astype("float32"); X_val_i=imp.transform(X_val).astype("float32"); X_adapt_i=imp.transform(X_adapt).astype("float32"); X_eval_i=imp.transform(X_eval).astype("float32")
print("adapt",len(X_adapt),"eval",len(X_eval),"eval attack rate",y_eval.mean())

In [ ]:
# Corrected CORAL
def coral_fit(Xs,Xt):
    Xs=np.asarray(Xs,float); Xt=np.asarray(Xt,float); ms=Xs.mean(0); mt=Xt.mean(0); Cs=np.cov(Xs-ms,rowvar=False)+1e-6*np.eye(Xs.shape[1]); Ct=np.cov(Xt-mt,rowvar=False)+1e-6*np.eye(Xt.shape[1])
    def mp(M,p):
        v,U=np.linalg.eigh(M); v=np.clip(v,1e-12,None); return U@np.diag(v**p)@U.T
    return {"ms":ms,"mt":mt,"A":mp(Cs,-.5)@mp(Ct,.5)}
def coral_source(X,c): return ((np.asarray(X)-c["ms"])@c["A"]+c["mt"]).astype("float32")
def cd(a,b): return float(np.linalg.norm(np.cov(a,rowvar=False)-np.cov(b,rowvar=False),ord="fro"))
cor=coral_fit(X_fit_i,X_adapt_i); X_fit_c=coral_source(X_fit_i,cor); X_val_c=coral_source(X_val_i,cor); X_adapt_c=X_adapt_i.astype("float32"); X_eval_c=X_eval_i.astype("float32"); print("covariance before",cd(X_fit_i,X_adapt_i)); print("covariance after",cd(X_fit_c,X_adapt_c))

In [ ]:
def eval_bin(y,p,t=.5):
    pred=(p>=t).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel(); return {"threshold":float(t),"accuracy":accuracy_score(y,pred),"precision":precision_score(y,pred,zero_division=0),"recall":recall_score(y,pred,zero_division=0),"f1":f1_score(y,pred,zero_division=0),"macro_f1":f1_score(y,pred,average="macro",zero_division=0),"pr_auc":average_precision_score(y,p),"roc_auc":roc_auc_score(y,p),"fpr":fp/(fp+tn) if fp+tn else np.nan,"tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),"predicted_attack_rate":float(pred.mean())}
def choose_thr(y,p):
    ts=np.linspace(.05,.95,181); fs=[f1_score(y,p>=t,zero_division=0) for t in ts]; return float(ts[int(np.argmax(fs))])
rf=RandomForestClassifier(n_estimators=250,random_state=RANDOM_STATE,n_jobs=-1,max_features="sqrt",min_samples_leaf=2,class_weight="balanced"); t0=time.perf_counter(); rf.fit(X_fit_c,y_fit); train_seconds=time.perf_counter()-t0; p_val=rf.predict_proba(X_val_c)[:,1]; source_thr=choose_thr(y_val,p_val); print("train sec",train_seconds,"source threshold",source_thr); display(pd.DataFrame([eval_bin(y_val,p_val,source_thr)]))

## BBSE estimation

BBSE uses the source validation confusion behavior and predictions on **unlabeled** CICIDS adaptation data to estimate the target class prior. Then it corrects posterior odds. This is a prior-shift hypothesis; the final frozen evaluation determines whether it helps.

In [ ]:
def bbse(y_src,p_src,p_tgt,threshold=.5,ridge=1e-6):
    hs=(p_src>=threshold).astype(int); ht=(p_tgt>=threshold).astype(int); C=np.zeros((2,2))
    for j in (0,1):
        m=y_src==j; C[:,j]=np.bincount(hs[m],minlength=2)/m.sum()
    q=np.bincount(ht,minlength=2).astype(float); q/=q.sum(); Cr=C+ridge*np.eye(2); pi=np.linalg.pinv(Cr)@q; pi=np.maximum(pi,0); pi/=pi.sum(); return C,q,pi,float(np.linalg.cond(Cr))
p_adapt=rf.predict_proba(X_adapt_c)[:,1]; C_source,q_target,pi_t,cond=bbse(y_val,p_val,p_adapt); pi_s=np.bincount(y_fit,minlength=2).astype(float); pi_s/=pi_s.sum(); odds_ratio=(pi_t[1]/pi_t[0])/(pi_s[1]/pi_s[0]); print("source confusion behavior\n",C_source); print("unlabeled target predicted distribution",q_target); print("estimated target prior",pi_t); print("source prior",pi_s); print("prior odds ratio",odds_ratio); print("condition number",cond); print("post-hoc actual adaptation prior (NOT USED)",np.bincount(y_adapt_hidden,minlength=2)/len(y_adapt_hidden))

In [ ]:
# Apply correction to frozen CICIDS evaluation
p_eval_source=rf.predict_proba(X_eval_c)[:,1]; source_odds=p_eval_source/np.maximum(1-p_eval_source,1e-12); p_eval_bbse=(source_odds*odds_ratio)/(1+source_odds*odds_ratio)
source_thr_odds=source_thr/max(1-source_thr,1e-12); bbse_thr_odds=source_thr_odds*odds_ratio; bbse_thr=bbse_thr_odds/(1+bbse_thr_odds)
print("source threshold",source_thr,"BBSE threshold",bbse_thr)

In [ ]:
# Final frozen evaluation — target labels are read only here
rows=[]; a=eval_bin(y_eval,p_eval_source,source_thr); a["method"]="CORAL + balanced"; rows.append(a); b=eval_bin(y_eval,p_eval_bbse,bbse_thr); b["method"]="CORAL + balanced + BBSE"; rows.append(b); c=eval_bin(y_eval,p_eval_bbse,.5); c["method"]="CORAL + balanced + BBSE @ 0.50"; rows.append(c); results=pd.DataFrame(rows); display(results[["method","threshold","accuracy","precision","recall","f1","macro_f1","pr_auc","roc_auc","fpr","tn","fp","fn","tp","predicted_attack_rate"]].sort_values("f1",ascending=False))

In [ ]:
# Direct comparison
base=results[results.method=="CORAL + balanced"].iloc[0]; new=results[results.method=="CORAL + balanced + BBSE"].iloc[0]; comparison=pd.DataFrame({"metric":["F1","Macro-F1","PR-AUC","ROC-AUC","Precision","Recall","FPR"],"CORAL + balanced":[base.f1,base.macro_f1,base.pr_auc,base.roc_auc,base.precision,base.recall,base.fpr],"CORAL + balanced + BBSE":[new.f1,new.macro_f1,new.pr_auc,new.roc_auc,new.precision,new.recall,new.fpr]}); comparison["BBSE_minus_benchmark"]=comparison["CORAL + balanced + BBSE"]-comparison["CORAL + balanced"]; display(comparison); print("F1 change",new.f1-base.f1)

In [ ]:
# Save outputs
results.to_csv(TABLES/"01_phase19_bbse_results.csv",index=False); comparison.to_csv(TABLES/"02_phase19_bbse_comparison.csv",index=False); pd.DataFrame({"class":["benign","attack"],"source_prior":pi_s,"estimated_target_prior":pi_t,"prior_ratio":pi_t/np.maximum(pi_s,1e-12)}).to_csv(TABLES/"03_phase19_prior_estimate.csv",index=False); config={"phase":19,"method":"BBSE_unsupervised_target_prior_correction","random_state":RANDOM_STATE,"target_pool_n":len(cicids_pool),"target_adaptation_n":len(X_adapt),"target_evaluation_n":len(X_eval),"features":FEATURES,"source_threshold":float(source_thr),"bbse_threshold":float(bbse_thr),"source_prior":pi_s.tolist(),"estimated_target_prior":pi_t.tolist(),"prior_odds_ratio":float(odds_ratio),"condition_number":float(cond),"target_labels_used_for_adaptation":False,"target_labels_used_for_threshold_selection":False}; json.dump(config,open(TABLES/"04_phase19_config.json","w",encoding="utf-8"),indent=2); print("Saved outputs to",TABLES)

## Decision rule

If BBSE improves F1 while reducing FPR, keep it. If it improves PR-AUC/ROC-AUC but not F1, ranking improved but conditional shift/calibration remains. If it does not improve, move to a conditional domain-adaptation method. Do not tune BBSE using CICIDS labels.